In [ ]:
MODE = 'PDE' # Options: 'PDE', 'PFE', 'PDE360'

In [2]:
import os
import pandas as pd
import glob

# Read all match files to extract home team names
if MODE == 'PDE360':
    files = glob.glob("sb_offline_360_data/events_*.pkl")
elif MODE in ('PDE', 'PFE'):
    files = glob.glob("sb_offline_data/match_*.pkl")

home_teams_dict = {}
match_ids = []
missing = 0

for file in files:
    # Get match ID from filename
    m_id = int(os.path.basename(file).split('_')[1].split('.')[0])
    match_ids.append(m_id)
    
    # Load events
    events = pd.read_pickle(file)

    # Extract the home team from the Starting XI events
    starting_xis = events[events['type'] == 'Starting XI']
    if not starting_xis.empty:
        home_team = starting_xis.iloc[0]['team']
        home_teams_dict[m_id] = home_team
    else:
        home_teams_dict[m_id] = "Unknown"
        missing += 1

print(f"Processed {len(match_ids)} matches. Missing home team info for {missing} matches.")

Processed 1238 matches. Missing home team info for 0 matches.


In [3]:
# Load filtered event catalogue and raw events used for contamination checks.
if MODE == 'PDE':
    df = pd.read_csv('output/pde/catalogue/events_filtered.csv')
    events_all_df = pd.read_csv('output/pde/catalogue/events_raw.csv')
elif MODE == 'PDE360':
    df = pd.read_csv('output/pde360/catalogue/events_filtered.csv')
    events_all_df = pd.read_csv('output/pde360/catalogue/events_raw.csv')
elif MODE == 'PFE':
    df = pd.read_csv('output/pfe/catalogue/events_filtered.csv')
    events_all_df = pd.read_csv('output/pfe/catalogue/events_raw.csv')

In [4]:
def get_score_diff_from_cache(meta, team, t_seconds):

    # Calculate score difference using pre-aggregated data in meta before t_seconds of the match for the given team
    # Goals and own goals that benefited the team
    team_scored = sum(1 for g in meta['goals'] if g['total_seconds'] < t_seconds and g['team'] == team)
    team_benefited = sum(1 for og in meta['own_goals'] if og['total_seconds'] < t_seconds and og['team'] != team)
    
    opp_scored = sum(1 for g in meta['goals'] if g['total_seconds'] < t_seconds and g['team'] != team)
    opp_benefited = sum(1 for og in meta['own_goals'] if og['total_seconds'] < t_seconds and og['team'] == team)
    
    return (team_scored + team_benefited) - (opp_scored + opp_benefited)

In [5]:
import random
import zlib

def find_control_windows(event_row, events_all_df, all_match_ids, match_metadata_cache, required_controls=3, window_seconds=300, random_seed=42):
    controls = []
    
    target_t = event_row['total_seconds']
    target_diff = event_row['goal_diff']
    period = event_row['period']
    event_match_id = event_row['match_id']
    event_id = event_row['event_id']
    
    # Deterministic but event-specific shuffle: reproducible runs with varied match order across PDEs.
    event_token = f"{event_id}_{event_match_id}_{period}_{target_t}"
    event_seed = (random_seed + zlib.crc32(event_token.encode('utf-8'))) % (2**32)
    other_matches = [m for m in all_match_ids if m != event_match_id]
    random.Random(event_seed).shuffle(other_matches)
    
    for c_match_id in other_matches:
        if len(controls) >= required_controls:
            break # If we've already found enough controls, stop searching
            
        if c_match_id not in match_metadata_cache:
            continue
            
        c_match_events = events_all_df[events_all_df['match_id'] == c_match_id]
        meta = match_metadata_cache[c_match_id]
        
        # Possible control points within the window around target_t
        clean_candidates = []
        for offset in range(-window_seconds, window_seconds + 1, 30): # 30 second steps
            t_cand = target_t + offset
            
            conflicts = c_match_events[
                (c_match_events['total_seconds'] >= t_cand - window_seconds) & 
                (c_match_events['total_seconds'] <= t_cand + window_seconds)
            ]
            if conflicts.empty:
                clean_candidates.append(t_cand)
                
        # If there are no clean candidates in this match, move to the next one
        if not clean_candidates:
            continue
            
        # Load match events to check period boundaries and calculate goal difference at candidate times
        if period not in meta['periods']:
            continue
            
        p_start = meta['periods'][period]['min']
        p_end = meta['periods'][period]['max']
        
        for t_cand in clean_candidates:
            # Window size check
            if (t_cand - p_start < window_seconds) or (p_end - t_cand < window_seconds):
                continue 
                
            # Check goal difference
            valid_team = None
            for team in meta['teams']:
                if get_score_diff_from_cache(meta, team, t_cand) == target_diff:
                    valid_team = team
                    break
                    
            # If we found a valid team with the same goal difference at this candidate time, we can create a control window
            if valid_team:
                control_row = {
                    'control_id': f"CTRL_{event_row['event_id']}_{len(controls)+1}",
                    'match_id': c_match_id,
                    'period': period,
                    'total_seconds': t_cand,
                    'goal_diff': target_diff,
                    'event_type': 'Control Window',
                    'affected_team': valid_team,
                    'paired_event_id': event_row['event_id'],
                }
                controls.append(control_row)
                break # Go to next match
                
    return controls

In [6]:
match_metadata = {}

# Making a match metadata cache for quick access during control window search
for i, m_id in enumerate(match_ids):

    if MODE == 'PDE360':
        events = pd.read_pickle(f"sb_offline_360_data/events_{m_id}.pkl")
    elif MODE in ('PDE', 'PFE'):
        events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    
    events['total_seconds'] = events['minute'] * 60 + events['second']
    
    # Period boundaries
    periods = events.groupby('period')['total_seconds'].agg(['min', 'max']).to_dict('index')
    
    # Goal and own goal extraction
    if 'shot_outcome' in events.columns:
        goals = events[(events['type'] == 'Shot') & (events['shot_outcome'] == 'Goal')]
        goals_list = goals[['total_seconds', 'team']].to_dict('records')
    else:
        goals_list = []
        
    own_goals = events[events['type'] == 'Own Goal Against']
    own_goals_list = own_goals[['total_seconds', 'team']].to_dict('records')
    
    match_metadata[m_id] = {
        'periods': periods,
        'goals': goals_list,
        'own_goals': own_goals_list,
        'teams': events['team'].unique().tolist()
    }

        
    if (i + 1) % 50 == 0:
        print(f"  ... {i + 1}/{len(match_ids)} matches loaded.")

  ... 50/1238 matches loaded.
  ... 100/1238 matches loaded.
  ... 150/1238 matches loaded.
  ... 200/1238 matches loaded.
  ... 250/1238 matches loaded.
  ... 300/1238 matches loaded.
  ... 350/1238 matches loaded.
  ... 400/1238 matches loaded.
  ... 450/1238 matches loaded.
  ... 500/1238 matches loaded.
  ... 550/1238 matches loaded.
  ... 600/1238 matches loaded.
  ... 650/1238 matches loaded.
  ... 700/1238 matches loaded.
  ... 750/1238 matches loaded.
  ... 800/1238 matches loaded.
  ... 850/1238 matches loaded.
  ... 900/1238 matches loaded.
  ... 950/1238 matches loaded.
  ... 1000/1238 matches loaded.
  ... 1050/1238 matches loaded.
  ... 1100/1238 matches loaded.
  ... 1150/1238 matches loaded.
  ... 1200/1238 matches loaded.


In [7]:
all_controls = []

for idx, event_row in df.iterrows():
    found_controls = find_control_windows(
        event_row=event_row,
        events_all_df=events_all_df,
        all_match_ids=match_ids,
        match_metadata_cache=match_metadata,
        required_controls=3
    )
    all_controls.extend(found_controls)
    
    if (idx + 1) % 200 == 0:
        print(f"[{idx + 1}/{len(df)}] Event processed...")

controls_df = pd.DataFrame(all_controls)

if MODE == 'PDE':
    controls_df.to_csv('output/pde/controls/controls.csv', index=False, encoding='utf-8-sig')
elif MODE == 'PDE360':
    controls_df.to_csv('output/pde360/controls/controls.csv', index=False, encoding='utf-8-sig')
elif MODE == 'PFE':
    controls_df.to_csv('output/pfe/controls/controls.csv', index=False, encoding='utf-8-sig')

[200/3006] Event processed...
[400/3006] Event processed...
[600/3006] Event processed...
[800/3006] Event processed...
[1000/3006] Event processed...
[1200/3006] Event processed...
[1400/3006] Event processed...
[1600/3006] Event processed...
[1800/3006] Event processed...
[2000/3006] Event processed...
[2200/3006] Event processed...
[2400/3006] Event processed...
[2600/3006] Event processed...
[2800/3006] Event processed...
[3000/3006] Event processed...


In [8]:
def get_metrics(df, target_team, window_sec = 300):
        
        if df.empty or window_sec <= 0:
            return {m: 0.0 for m in ['pass_acc_ratio', 'fwd_pass_ratio', 'long_ball_ratio', 'xg_rate', 
                                    'shot_rate', 'avg_xg', 'dribble_rate', 'turnover_rate', 
                                    'foul_rate', 'press_intensity']}
        
        team_df = df[df.team == target_team].copy()
        
        # Filter out non-open play events for passing and shooting metrics
        if 'pass_type' in team_df.columns:
            team_df = team_df[~team_df.pass_type.isin(['Corner', 'Free Kick', 'Goal Kick', 'Throw-in'])]
        if 'shot_type' in team_df.columns:
            team_df = team_df[~team_df.shot_type.isin(['Penalty', 'Free Kick'])]

        res = {}
        
        # Passing Metrics
        passes = team_df[team_df.type == 'Pass']
        n_passes = len(passes)
        if n_passes > 0:
            # Pass completion rate: Successful / Total
            res['pass_acc_ratio'] = len(passes[passes.pass_outcome.isnull()]) / n_passes
            
            # Forward pass ratio: Forward / Total
            res['fwd_pass_ratio'] = (passes['end_x'] > passes['x']).sum() / n_passes
            
            # Long ball ratio: Long (>32m) / Total
            distances = np.sqrt((passes['end_x'] - passes['x'])**2 + (passes['end_y'] - passes['y'])**2)
            res['long_ball_ratio'] = (distances > 32).sum() / n_passes
        else:
            res['pass_acc_ratio'] = res['fwd_pass_ratio'] = res['long_ball_ratio'] = 0.0

        # Shooting Metrics
        shots = team_df[team_df.type == 'Shot']
        res['xg_rate'] = shots.shot_statsbomb_xg.sum()
        res['shot_rate'] = len(shots)
        res['avg_xg'] = shots.shot_statsbomb_xg.mean() if len(shots) > 0 else 0.0

        # Dribbling and Turnover Metrics
        res['dribble_rate'] = len(team_df[team_df.type == 'Dribble'])
        res['turnover_rate'] = len(team_df[team_df.type.isin(['Dispossessed', 'Miscontrol'])])

        # Defensive Metrics
        res['foul_rate'] = len(team_df[team_df.type == 'Foul Committed'])
        res['press_intensity'] = len(team_df[team_df.type == 'Pressure'])
        
        return res

In [9]:
def calculate_single_window_metrics(row, events, home_team_name, W_seconds=300):

    t = row['total_seconds']
    team = row['affected_team']
    period = row['period']
    
    # Define pre and post windows considering half-time boundaries
    pre_events = events[(events.total_seconds >= t - W_seconds) & 
                        (events.total_seconds < t) & 
                        (events.period == period)]
    
    post_events = events[(events.total_seconds > t) & 
                         (events.total_seconds <= t + W_seconds) & 
                         (events.period == period)]

    # Calculate metrics for both windows
    pre_m = get_metrics(pre_events, team)
    post_m = get_metrics(post_events, team)

    # Combine into a single row
    output = row.to_dict()
    for metric in pre_m.keys():
        output[f'pre_{metric}'] = pre_m[metric]
        output[f'post_{metric}'] = post_m[metric]
        output[f'delta_{metric}'] = post_m[metric] - pre_m[metric]
    
    # Home/Away indicator
    output['is_home'] = 1 if team == home_team_name else 0

    # Compute red card difference in the game
    events_before_t = events[events.total_seconds <= t]
    focal_reds = 0
    opp_reds = 0
    
    if 'foul_committed_card' in events.columns:
        reds = events_before_t[events_before_t['foul_committed_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds[reds.team == team])
        opp_reds += len(reds[reds.team != team])
        
    if 'bad_behaviour_card' in events.columns:
        reds_bb = events_before_t[events_before_t['bad_behaviour_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds_bb[reds_bb.team == team])
        opp_reds += len(reds_bb[reds_bb.team != team])
        
    output['red_card_diff'] = focal_reds - opp_reds

    # y_conceded: Check if the affected team concedes a goal in the next 10 minutes
    events_next_10m = events[(events.total_seconds > t) & (events.total_seconds <= t + 600)]
    
    target_met = 0
    if 'shot_outcome' in events_next_10m.columns:
        if MODE == 'PDE':
            # PDE: Did the team concede a goal? (The opposing team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team != team)]
        elif MODE in ('PFE', 'PDE360'):
            # PFE/PDE360: Did the team score a goal? (The examined team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team == team)]
            
        if len(goals) > 0:
            target_met = 1

    output['y'] = target_met
        
    return output

In [10]:
from math import e

import numpy as np
from tqdm import tqdm

final_control_metrics = []
grouped_controls = controls_df.groupby('match_id')

for m_id, match_controls in tqdm(grouped_controls, desc="Processing Matches", unit="matches"):
    current_home_team = home_teams_dict.get(m_id, "Unknown")
    
    if MODE == 'PDE360':
        events = pd.read_pickle(f"sb_offline_360_data/events_{m_id}.pkl")
    elif MODE in ('PDE', 'PFE'):
        events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")

    events['total_seconds'] = events['minute'] * 60 + events['second']
    events['x'] = events['location'].str[0]
    events['y'] = events['location'].str[1]
    
    if 'pass_end_location' in events.columns:
        events['end_x'] = events['pass_end_location'].str[0]
        events['end_y'] = events['pass_end_location'].str[1]
    else:
        events['end_x'] = np.nan
        events['end_y'] = np.nan
        
    for _, control_row in match_controls.iterrows():
        final_control_metrics.append(calculate_single_window_metrics(control_row, events, current_home_team))

controls_with_metrics_df = pd.DataFrame(final_control_metrics)

if MODE == 'PDE':
    controls_with_metrics_df.to_csv('output/pde/controls/controls_with_metrics.csv', index=False, encoding='utf-8-sig')
elif MODE == 'PDE360':
    controls_with_metrics_df.to_csv('output/pde360/controls/controls_with_metrics.csv', index=False, encoding='utf-8-sig')
elif MODE == 'PFE':
    controls_with_metrics_df.to_csv('output/pfe/controls/controls_with_metrics.csv', index=False, encoding='utf-8-sig')

Processing Matches: 100%|██████████| 1233/1233 [08:40<00:00,  2.37matches/s]
